## <span style="color: teal;">Exporting MODIS Column Water Vapor (WV) for Greece using Google Earth Engine </span>

### <span style="color: teal;">Initialize Google Earth Engine & Define Study Area</span>

In [ ]:
import ee

# Authenticate and initialize Earth Engine
ee.Authenticate()
ee.Initialize()

# Define the area of interest (Greece)
greece = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017").filter(ee.Filter.eq('country_na', 'Greece')).geometry()

### <span style="color: teal;">Function to Export Daily WV Images</span>

In [ ]:
def export_wv_image(year, month, day):
    """Exports a MODIS Column_WV image for a specific day."""
    start_date = ee.Date.fromYMD(year, month, day)
    end_date = start_date.advance(1, 'day')

    # Get MODIS Column_WV for the day
    daily_image = ee.ImageCollection('MODIS/061/MCD19A2_GRANULES') \
                   .filterDate(start_date, end_date) \
                   .select('Column_WV') \
                   .mean() \
                   .multiply(0.001)  # Apply scale factor

    # Check if the image contains valid data
    band_value = daily_image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=greece,
        scale=1000,
        maxPixels=1e13
    ).get('Column_WV')

    if band_value.getInfo() is not None:
        # Clip to Greece and export
        task = ee.batch.Export.image.toDrive(
            image=daily_image.clip(greece),
            description=f'ColumnWV_{year}_{month:02d}_{day:02d}',
            folder='WV_Exports',
            fileNamePrefix=f'ColumnWV_{year}_{month:02d}_{day:02d}',
            region=greece,
            scale=1000,
            crs='EPSG:4326',
            maxPixels=1e13
        )
        task.start()
        print(f'Exporting: ColumnWV_{year}-{month:02d}-{day:02d}')
    else:
        print(f'No data for: {year}-{month:02d}-{day:02d}')


### <span style="color: teal;">Select Year Range & Process Data</span>

In [ ]:
import calendar

# Set the years to process
years = [2017, 2018, 2019]  # Modify this list to choose specific years

# Loop through selected years and export data
for year in years:
    for month in range(1, 13):
        for day in range(1, calendar.monthrange(year, month)[1] + 1):
            export_wv_image(year, month, day)
